# FINS notebook 5 — learned operators under noise (Colab, GPU)

Registered in `README_EXPERIMENTS.md` before any of these runs (EX4, 07.10; LW4 and LW5, 08.10):
* cell 2: FINS weak through the protocol's heat CNOs, clean (EX4; notebook 4's runs, skipped if done), and through
  exp48's heat CNOs retrained on 1e-3 and 1e-2 data, on equally noisy evaluation data (LW4); 9 runs.
* cell 3: FNO steppers for Benjamin–Ono and Kuramoto–Sivashinsky trained on 1e-2-corrupted 'mixed' data, seeds
  42–44 (exp39's recipe, ~15–25 min each), each checked by pointwise FINS on data seed 1007.
* cell 4: FINS weak and pointwise through those steppers on equally noisy evaluation data (LW5); 12 runs.
* cell 5 shows progress; cell 6 zips the outputs and the new steppers into `MyDrive/fins_nb5.zip`.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down). The heat
CNOs (`S_theta_heat_s{s}_g64x128_cno_smooth_L1.pt` and `..._smooth_L1_n1e-3.pt`, `..._n1e-2.pt`) are on the Drive from
the protocol and exp48 runs; cell 1 finds them. After a disconnect rerun cell 1 and the interrupted cell: finished
runs are skipped and a stepper training resumes from its partial state.

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp56), the heat CNOs, dependencies
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(R, exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp56(z):
    with zipfile.ZipFile(z) as zz:
        return 'fins_repo/experiments/exp56_jobs.py' in zz.namelist()


ZIP_PATH = newest('fins_repo_full*.zip', has_exp56)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp56: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
for s in (42, 43, 44):
    for tag in ('', '_n1e-3', '_n1e-2'):
        name = f'S_theta_heat_s{s}_g64x128_cno_smooth_L1{tag}.pt'
        dst = os.path.join(R, name)
        if os.path.exists(dst):
            continue
        hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
        if not hits:
            raise SystemExit(f'{name} is not on the Drive (protocol / exp48 heat CNO)')
        shutil.copy(hits[0], dst)
        print('copied', hits[0], '->', dst)
print('heat CNOs present')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator', 'torchdiffeq'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')
WEAK = ['--K', '16', '--stride', '1', '--wt', '2', '--wp', '2', '--wx', 'auto', '--wx-factor', '2']


def go(out, cmd):
    if os.path.exists(out):
        print('done', os.path.relpath(out, R))
        return
    os.makedirs(os.path.join(os.path.dirname(out), 'logs'), exist_ok=True)
    log = os.path.join(os.path.dirname(out), 'logs', os.path.basename(out).replace('.npz', '.log'))
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    keep = [l.strip() for l in open(log).read().splitlines() if 'algebra inside' in l or 'own count' in l]
    print(os.path.relpath(out, R), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:120] for k in keep[-2:]], sep='\n   ')

In [ ]:
# 2. heat: FINS weak through the clean protocol CNOs (EX4) and through exp48's CNOs retrained on noisy data (LW4)
for cond, sigma, tag in (('clean', '0', '_smooth_L1'), ('n1e-3', '0.001', '_smooth_L1_n1e-3'),
                         ('n1e-2', '0.01', '_smooth_L1_n1e-2')):
    for s in (42, 43, 44):
        out = os.path.join(R, 'noise_weak_runs', cond, f'weak-learned_heat_s{s}.npz')
        go(out, ['exp49_weak.py', '--pde', 'heat', '--op', 'learned', '--seed', str(s), '--data-seed', str(1000 + s),
                 '--tag', tag, '--noise', sigma, *WEAK, '--save', out])

In [ ]:
# 3. BO and KS steppers trained on 1e-2-corrupted 'mixed' data, each checked by pointwise FINS on data seed 1007
os.makedirs(os.path.join(R, 'steppers'), exist_ok=True)
for pde in ('bo', 'ks'):
    for s in (42, 43, 44):
        out = os.path.join(R, 'steppers', f'pointwise_{pde}_mixed_stepper_n1e-2_s{s}.npz')
        go(out, ['exp39_localised.py', '--pde', pde, '--op', 'stepper', '--train-ic', 'mixed', '--train-if-missing',
                 '--tag', '_n1e-2', '--noise', '0.01', '--seed', str(s), '--data-seed', '1007', '--ic', 'periodic',
                 '--n', '10', '--K', '16', '--every', '16', '--interior', '--ut', 'fd', '--save', out])

In [ ]:
# 4. FINS weak and pointwise through the noisy-data steppers, on equally noisy evaluation data (LW5)
for pde in ('bo', 'ks'):
    d = os.path.join(R, f'{pde}_runs', 'n1e-2')
    for s in (42, 43, 44):
        ds = ['--seed', str(s), '--data-seed', str(1000 + s)]
        go(os.path.join(d, f'weak-learned_{pde}_s{s}.npz'),
           ['exp49_weak.py', '--pde', pde, '--op', 'learned', *ds, '--tag', '_n1e-2', '--noise', '0.01', *WEAK,
            '--save', os.path.join(d, f'weak-learned_{pde}_s{s}.npz')])
        go(os.path.join(d, f'pointwise-learned_{pde}_s{s}.npz'),
           ['exp39_localised.py', '--pde', pde, '--op', 'stepper', '--train-ic', 'mixed', '--tag', '_n1e-2', *ds,
            '--ic', 'periodic', '--n', '10', '--K', '16', '--every', '16', '--thin', '1', '--interior', '--ut', 'fd',
            '--noise', '0.01', '--save', os.path.join(d, f'pointwise-learned_{pde}_s{s}.npz')])

In [ ]:
# 5. progress
want = [os.path.join(R, 'noise_weak_runs', c, f'weak-learned_heat_s{s}.npz') for c in ('clean', 'n1e-3', 'n1e-2')
        for s in (42, 43, 44)]
want += [os.path.join(R, f'{p}_runs', 'n1e-2', f'{k}-learned_{p}_s{s}.npz') for p in ('bo', 'ks')
         for k in ('weak', 'pointwise') for s in (42, 43, 44)]
want += [os.path.join(R, f'S_theta_{p}_s{s}_mixed_stepper_n1e-2.pt') for p in ('bo', 'ks') for s in (42, 43, 44)]
done = [w for w in want if os.path.exists(w)]
print(f'{len(done)} / {len(want)} done')
for w in want:
    if w not in done:
        print('missing', os.path.relpath(w, R))

In [ ]:
# 6. zip the outputs and the new steppers: MyDrive/fins_nb5.zip
out = '/content/drive/MyDrive/fins_nb5.zip'
files = []
for c in ('clean', 'n1e-3', 'n1e-2'):
    d = os.path.join(R, 'noise_weak_runs', c)
    files += glob.glob(os.path.join(d, 'weak-learned_heat_s4*.npz')) + glob.glob(os.path.join(d, 'logs', 'weak-learned_heat_s4*.log'))
for p in ('bo', 'ks'):
    d = os.path.join(R, f'{p}_runs', 'n1e-2')
    files += glob.glob(os.path.join(d, '*-learned_*.npz')) + glob.glob(os.path.join(d, 'logs', '*-learned_*.log'))
    files += glob.glob(os.path.join(R, f'S_theta_{p}_s4*_mixed_stepper_n1e-2.pt'))
    files += glob.glob(os.path.join(R, 'steppers', f'pointwise_{p}_mixed_stepper_n1e-2_s4*'))
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(set(files)):
        z.write(f, os.path.relpath(f, R))
print(out, os.path.getsize(out), 'bytes;', len(set(files)), 'files')